# Alternative-scaled multinomial logit

## Model

Alternative-specific scales allow the response to systematic utility to
differ across modes:

$$
P_{nj}=
\frac{\exp(V_{nj}/s_j)}
{\sum_{k\in\mathcal{C}_n}\exp(V_{nk}/s_k)},\qquad s_j>0.
$$

### Notation

- $N$ is the total number of choice observations, and $n=1,\ldots,N$ indexes
  one observation. The symbols $j$ and $k$ index alternatives in its choice
  set $\mathcal{C}_n$.
- $P_{nj}$ is the probability that observation $n$ selects alternative $j$.
- $V_{nj}$ is the unscaled systematic utility. The example uses TRAIN and CAR
  constants plus generic `time` and `cost` coefficients; SM is the utility
  reference.
- $s_j>0$ is the alternative-specific utility scale. Dividing by a larger
  $s_j$ makes a given utility difference contribute less sharply to choice
  probabilities.
- `SCALE_TRAIN` and `SCALE_CAR` estimate
  $s_{\mathrm{TRAIN}}$ and $s_{\mathrm{CAR}}$.
  `SCALE_SM` fixes $s_{\mathrm{SM}}=1$ to identify the relative scales.
- The denominator evaluates every alternative with its own scale before
  normalizing the probabilities to sum to one.

Utility and scale parameters are estimated jointly by maximum likelihood.

### Execution environment

Machine configuration: AMD Ryzen 9 9950X3D CPU (16 cores), 64 GB RAM, and
NVIDIA GeForce RTX 5090 GPU (32 GB VRAM), running Ubuntu 24.04.4 with
PyTorch 2.12.1 and CUDA 13.0. Install the released package with
`pip install torchdcm`, then select CPU or CUDA through the `device` argument.


In [1]:
import torch

from IPython.display import HTML, display

from torchdcm import (
    AlternativeScale,
    Beta,
    ChoiceDataset,
    ScaledMultinomialLogit,
    UtilitySpec,
)
from torchdcm.datasets import make_swissmetro_like

# Use double precision and a fixed seed for stable, reproducible estimation.
torch.set_default_dtype(torch.float64)
torch.manual_seed(7)
# Use CUDA automatically when available; set device = "cpu" to force CPU.
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"TorchDCM example running on {device}")


TorchDCM example running on cuda


In [2]:
# Helper: map wide attributes and availability into a ChoiceDataset.
def make_choice_data(n_obs=180, seed=7, *, observation_variables=None):
    frame = make_swissmetro_like(n_obs=n_obs, seed=seed)
    data = ChoiceDataset.from_wide(
        frame,
        alternatives=["TRAIN", "SM", "CAR"],
        choice="choice",
        variables={
            "time": {
                "TRAIN": "time_train",
                "SM": "time_sm",
                "CAR": "time_car",
            },
            "cost": {
                "TRAIN": "cost_train",
                "SM": "cost_sm",
                "CAR": "cost_car",
            },
        },
        obs_variables=observation_variables,
        availability={
            "TRAIN": "avail_train",
            "SM": "avail_sm",
            "CAR": "avail_car",
        },
        individual_id="person_id",
    )
    return frame, data


# Helper: define generic time and cost effects with SM as reference.
def make_utility_spec(suffix=""):
    tag = f"_{suffix}" if suffix else ""
    spec = UtilitySpec()
    spec.utility(
        "TRAIN",
        Beta(f"ASC_TRAIN{tag}", init=0.0)
        + Beta(f"B_TIME{tag}", init=-0.01) * "time"
        + Beta(f"B_COST{tag}", init=-0.10) * "cost",
    )
    spec.utility(
        "SM",
        Beta(f"B_TIME{tag}", init=-0.01) * "time"
        + Beta(f"B_COST{tag}", init=-0.10) * "cost",
    )
    spec.utility(
        "CAR",
        Beta(f"ASC_CAR{tag}", init=0.0)
        + Beta(f"B_TIME{tag}", init=-0.01) * "time"
        + Beta(f"B_COST{tag}", init=-0.10) * "cost",
    )
    return spec


# Create the estimation sample and shared utility specification.
frame, data = make_choice_data(seed=16)
spec = make_utility_spec()

In [3]:
# Estimate TRAIN and CAR scales while fixing SM to one.
scales = {
    "TRAIN": AlternativeScale(init=0.90, name="SCALE_TRAIN"),
    "SM": AlternativeScale(init=1.00, fixed=True, name="SCALE_SM"),
    "CAR": AlternativeScale(init=1.10, name="SCALE_CAR"),
}
model = ScaledMultinomialLogit(
    spec,
    scales,
    device=device,
    max_iter=60,
)
result = model.fit(data)
# Fit utility and scale parameters jointly, then render the report.
display(HTML(result.report().to_html()))


Model family,ScaledMultinomialLogit
Estimation objective,Maximum log likelihood
TorchDCM version,0.1.0
PyTorch version,2.12.1+cu130
Python version,3.12.13
Operating system,Linux-6.17.0-35-generic-x86_64-with-glibc2.39
Device,cuda
Tensor dtype,float64
Optimizer,torch.optim.LBFGS
Maximum iterations,60
Line search,strong_wolfe
